# Đánh Giá (Validation) Mô Hình MulCo
Notebook này nạp cấu trúc mô hình, tải trọng số đã huấn luyện và tiến hành đánh giá trên tập Validation.

In [ ]:
import os
import sys
from pathlib import Path
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import transforms
from transformers import CLIPTokenizer, CLIPModel
from tqdm.notebook import tqdm
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

# Tự động tìm thư mục gốc chứa 'src'
current_dir = Path.cwd().resolve()
while current_dir.name and not (current_dir / 'src').exists():
    current_dir = current_dir.parent
PROJECT_ROOT = current_dir
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print(f"Project Root: {PROJECT_ROOT}")

from src.datasets.multimodal_raw_dataset import MultiModalRawDataset
from src.models.backbones.vision.convnext_cbam import ConvNeXt_CBAM
from src.models.fusion.mulco_fusion import MulCoFusionBlock
from src.models.multimodal.mulco_classifier import Conv1x1Classifier

In [ ]:
class MulCoEndToEnd(nn.Module):
    def __init__(self, num_classes=28, proj_dim=512, spatial_size=(7, 7)):
        super().__init__()
        self.image_backbone = ConvNeXt_CBAM(num_classes=num_classes)
        self.text_backbone = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").text_model
        self.img_proj = nn.Conv2d(1024, proj_dim, kernel_size=1)
        self.txt_proj = nn.Linear(512, proj_dim)
        self.fusion_blocks = nn.ModuleList([MulCoFusionBlock(dim=proj_dim, num_heads=8) for _ in range(3)])
        self.classifier = Conv1x1Classifier(in_channels=proj_dim, num_classes=num_classes, spatial_size=spatial_size)

    def forward(self, images, input_ids, attention_mask):
        img_feat = self.image_backbone.forward_features_spatial(images)
        txt_out = self.text_backbone(input_ids=input_ids, attention_mask=attention_mask)
        txt_feat = txt_out.last_hidden_state
        img_feat = self.img_proj(img_feat)
        txt_feat = self.txt_proj(txt_feat)
        
        for block in self.fusion_blocks:
            img_feat, txt_feat = block(img_feat, txt_feat)
            
        return self.classifier(img_feat)

In [ ]:
def custom_collate_fn(batch, tokenizer):
    images = torch.stack([b["image"] for b in batch])
    labels = torch.tensor([b["label"] for b in batch], dtype=torch.long)
    texts = [b["text"] for b in batch]
    text_tokens = tokenizer(texts, padding=True, truncation=True, max_length=77, return_tensors="pt")
    return images, text_tokens.input_ids, text_tokens.attention_mask, labels

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])
tokenizer = CLIPTokenizer.from_pretrained("openai/clip-vit-base-patch32")

val_dataset = MultiModalRawDataset(
    image_root=os.path.join(PROJECT_ROOT, "data/AIDG/dataset_PlantDoc/images/val"),
    caption_root=os.path.join(PROJECT_ROOT, "data/AIDG/captions_LLaVA/val"),
    transform=transform,
    use_depth_suppressed=True
)

val_loader = DataLoader(
    val_dataset, batch_size=16, shuffle=False, num_workers=2,
    collate_fn=lambda b: custom_collate_fn(b, tokenizer)
)

model = MulCoEndToEnd(num_classes=28).to(device)
# model.load_state_dict(torch.load("path/to/checkpoint.pth"))
model.eval()

all_preds = []
all_labels = []

with torch.no_grad():
    for images, input_ids, attn_mask, labels in tqdm(val_loader, desc="Validating"):
        images, input_ids, attn_mask = images.to(device), input_ids.to(device), attn_mask.to(device)
        logits = model(images, input_ids, attn_mask)
        preds = torch.argmax(logits, dim=1)
        
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.numpy())

acc = accuracy_score(all_labels, all_preds)
precision, recall, f1, _ = precision_recall_fscore_support(all_labels, all_preds, average="weighted", zero_division=0)

print(f"\nAccuracy : {acc:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1-Score : {f1:.4f}")